\*Please Run in colab

# Setup

### Environment Setup

In [1]:
!pip install instructor

In [2]:
# !cp /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train /content/
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 1913, done.
remote: Counting objects: 100% (620/620), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 1913 (delta 475), reused 443 (delta 438), pack-reused 1293 (from 2)
Receiving objects: 100% (1913/1913), 39.56 MiB | 14.56 MiB/s, done.
Resolving deltas: 100% (1218/1218), done.


In [4]:
# !cp /content/danidanou_Bloomberg_Financial_News_train /content/macro_financial_forecasting/applications/macro_financial_forecasting/data/danidanou_Bloomberg_Financial_News_train

In [5]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Mount GDrive

In [6]:
from google.colab import drive

# This will prompt you to authorize Colab to access your Google Drive.
drive.mount('/content/gdrive')
GDRIVE_PATH = "/content/gdrive/MyDrive/macro_financial_forecasting_files/"

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


### Code Setup

In [7]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")

train_data_loader = TrainDataLoader(config)

# Train Data Loader

In [8]:
print("Starting loading pipeline ...")
print(f"Config: {config}")

train_ds = train_data_loader.load()
print("Loading pipeline completed.")

Starting loading pipeline ...
Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'http

bloomberg_financial_data.parquet.gzip:   0%|          | 0.00/482M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/446762 [00:00<?, ? examples/s]


--- Download Successful! ---


Map:   0%|          | 0/446762 [00:00<?, ? examples/s]

Training dataset processed.

--- Starting validation of 446762 entries ---


Validating entries: 100%|██████████| 446762/446762 [00:29<00:00, 15003.55it/s]


--- Validation Complete! ---
Training dataset validated.
Saving processed dataset to local cache at '../data/danidanou_Bloomberg_Financial_News_train'...
Total number of rows: 446762
Loading pipeline completed.


# Data Processing Pipeline

Update these 2 variable to specify which indices to process.

In [9]:
DATA_START=0
DATA_END=100

In [10]:
from processor import NewsProcessor
import nest_asyncio
nest_asyncio.apply()

processor = NewsProcessor(config)
train_ds = processor.remove_redundant_info(train_ds[DATA_START:DATA_END])
df = processor.enrich_news_entries_with_classifications(train_ds, save_path=f"{GDRIVE_PATH}processed_news") #Sample size
df = processor.group_by_date_and_industry(df, save_path=f"{GDRIVE_PATH}grouped_news")
df = processor.filter_and_analyze_news(df)
df = processor.extract_impactful_news(df, top_n=3, save_path=f"{GDRIVE_PATH}impact_news")
df = processor.get_consolidated_sentiment(df, save_path=f"{GDRIVE_PATH}sentiment_news")

Device set to use cuda:0


Processing 100 news entries...


Industry Classification: 100%|██████████| 4/4 [00:05<00:00,  1.38s/batch]


Completed processing 100 entries

Dropped 2 (Industry, Date) pairs with Industry='None'
Remaining pairs: 19

Summary Statistics:
Total unique (Industry, Date) pairs: 19
Average articles per pair: 5.11
Max articles in a pair: 51
Min articles in a pair: 1
25th percentile: 1.0
50th percentile: 2.0
75th percentile: 4.5
Number of pairs with at least 3 articles: 6
Total articles: 97


Extracting top 3 impactful news per (Industry, Date) pair...


Processing groups: 100%|██████████| 19/19 [00:00<00:00, 44846.24it/s]


Processing 19 news entries...


FinBERT Sentiment: 100%|██████████| 1/1 [00:00<00:00,  7.44batch/s]

Completed processing 19 entries


In [11]:
df = await processor.get_explanation(df, save_path=f"{GDRIVE_PATH}sentiment_news")

Explanation: 100%|██████████| 19/19 [00:18<00:00,  1.04it/s]


In [12]:
df

,Industry,Date,News,ArticleCount,ImpactfulNews,AvgSentimentScore,SentimentScore,SentimentExplanation
0,Communication Services,2011-10-06,[{'Headline': 'FCC to Revamp Phone Subsidy to ...,2,[{'Headline': 'Euro-Area Leaders to Hold Summi...,0.709191,-0.291917,Overall sentiment for the Communications Servi...
1,Consumer Discretionary,2011-10-06,[{'Headline': 'PepsiCo May Purchase Russian Dr...,1,[{'Headline': 'PepsiCo May Purchase Russian Dr...,0.881740,0.888237,The article’s sentiment is strongly positive f...
2,Consumer Staples,2011-10-06,[{'Headline': 'Ukraine’s Grain Harvest Advance...,1,[{'Headline': 'Ukraine’s Grain Harvest Advance...,-0.918589,-0.917441,Explanation: FinBERT indicates a strongly nega...
3,Energy,2011-10-06,[{'Headline': 'Clean-Tech Companies Should Get...,9,[{'Headline': 'Norway Boosts Mongstad Carbon-S...,0.252093,-0.252850,"The energy-angle sentiment is mildly negative,..."
4,Financials,2011-10-06,[{'Headline': 'Ivory Coast Keeps Cocoa Export ...,51,[{'Headline': 'Remittances to Vietnam Thru Jul...,-0.306989,-0.032282,Combined FinBERT signal for the Financials set...
5,General Market,2011-10-06,[{'Headline': 'Farmland Seen Returning Up to 1...,4,[{'Headline': 'GE Study Finds Recession’s Job ...,-0.256293,-0.873807,Overall sentiment is negative: the combined Fi...
6,Health Care,2011-10-06,[{'Headline': 'House Panel Seeks Details on IR...,2,[{'Headline': 'Emdeon Said to Set Rate on $1.2...,0.666714,0.682417,Score: +0.67. Explanation: The Health Care new...
7,Industrials,2011-10-06,[{'Headline': 'Airbus German Workers Plan Work...,5,"[{'Headline': 'Polish Stocks: Getin, KGHM, Lot...",-0.296801,-0.868415,Explanation: The Industrials sentiment is nega...
8,Information Technology,2011-10-06,[{'Headline': 'Fans Hold IPhone-Lit Vigils for...,2,[{'Headline': 'Fans Hold IPhone-Lit Vigils for...,0.412332,0.002113,Net sentiment for the Information Technology t...
9,Materials,2011-10-06,[{'Headline': 'USDA Boxed Beef Cutout Closing ...,5,[{'Headline': 'Ukraine September Consumer Pric...,0.853697,0.875244,Combined materials-focused sentiment is strong...


In [13]:
import pandas as pd
pd.read_parquet(f"{GDRIVE_PATH}sentiment_news")

,Industry,Date,News,ArticleCount,ImpactfulNews,AvgSentimentScore,SentimentScore,SentimentExplanation
0,Communication Services,2011-10-06,[{'Article': 'U.S. regulators proposed overhau...,2,[{'Article': 'Euro-area government leaders wil...,0.709191,-0.291917,Overall sentiment for the Communications Servi...
1,Consumer Discretionary,2011-10-06,[{'Article': 'PepsiCo Inc. is in talks to buy ...,1,[{'Article': 'PepsiCo Inc. is in talks to buy ...,0.881740,0.888237,The article’s sentiment is strongly positive f...
2,Consumer Staples,2011-10-06,[{'Article': 'Ukraine’s grain harvest rose by ...,1,[{'Article': 'Ukraine’s grain harvest rose by ...,-0.918589,-0.917441,Explanation: FinBERT indicates a strongly nega...
3,Energy,2011-10-06,"[{'Article': 'Reed Hundt, head of the Coalitio...",9,[{'Article': 'Norway revised the cost estimate...,0.252093,-0.252850,"The energy-angle sentiment is mildly negative,..."
4,Financials,2011-10-06,[{'Article': 'Export taxes on cocoa beans from...,51,[{'Article': 'Remittances to Vietnam in the Ja...,-0.306989,-0.032282,Combined FinBERT signal for the Financials set...
5,General Market,2011-10-06,[{'Article': 'Farmland investments may return ...,4,[{'Article': 'Midsized U.S. companies that add...,-0.256293,-0.873807,Overall sentiment is negative: the combined Fi...
6,Health Care,2011-10-06,"[{'Article': 'Representative Charles Boustany,...",2,"[{'Article': 'Emdeon Inc. (EM) , the provider ...",0.666714,0.682417,Score: +0.67. Explanation: The Health Care new...
7,Industrials,2011-10-06,[{'Article': 'Airbus SAS labor unions in Germa...,5,[{'Article': 'Poland ’s WIG20 Index advanced f...,-0.296801,-0.868415,Explanation: The Industrials sentiment is nega...
8,Information Technology,2011-10-06,[{'Article': 'Apple Inc. (AAPL) fans worldwide...,2,[{'Article': 'Apple Inc. (AAPL) fans worldwide...,0.412332,0.002113,Net sentiment for the Information Technology t...
9,Materials,2011-10-06,[{'Article': 'October 6 (Bloomberg) -- This ta...,5,[{'Article': 'Following is a table detailing S...,0.853697,0.875244,Combined materials-focused sentiment is strong...
